# All together: one input, explicit objectives, frozen circuits

**Team attribution:** Alexandra Enachescu — objective variants/MEC; Carla Cotea — pruning, warm-start/multi-angle and information limits; Tudor-Constantin Pricop — benchmark engine/local search; Alexandru Picioroaga — hardware realization and mitigation studies. Source tips: `bfcb9d4`, `5be55f9`, `6933dde`, `bf51921`.

This is a small integration experiment, not a speedup claim. All methods receive the same reads. Full unpruned baseline pairwise cost **and full-read MEC** are common references. MEC is evaluated exactly; it is **not** substituted for the pairwise RZZ circuit Hamiltonian. Training uses local statevectors only; frozen circuits are then sampled ideally and on a fake-backend noise model. **Real QPU: not executed.** Recursive QAOA is not implemented. Standalone investigations and optional methods remain available in their original notebooks.

In [1]:
from pathlib import Path
adapter_path = Path('shared_adapters.ipynb')
if not adapter_path.exists():
    adapter_path = Path('bench') / 'shared_adapters.ipynb'
%run $adapter_path
import pandas as pd
from IPython.display import display
import importlib.metadata as metadata
print({name: metadata.version(name) for name in
       ('qiskit', 'qiskit-aer', 'qiskit-ibm-runtime', 'numpy', 'scipy', 'nbformat', 'nbclient')})


{'qiskit': '2.5.0', 'qiskit-aer': '0.17.2', 'qiskit-ibm-runtime': '0.51.0', 'numpy': '2.5.1', 'scipy': '1.18.0', 'nbformat': '5.11.1', 'nbclient': '0.11.0'}


## 1. Shared reads and exact reference

Default: the team's five-SNP puzzle (32 labels). Change only `READS` for another small input. Exact enumeration is capped here to prevent accidental large runs. No hidden truth is used in training or warm-start selection.

In [2]:
READS = tudor['toy_instance']().reads.copy()
N = READS.shape[1]
if N > 8 or N < 1 or not np.isin(READS, [-1, 0, 1]).all():
    raise ValueError('Smoke benchmark supports 1–8 SNPs with calls -1/0/1.')
STATES = tudor['bits'](np.arange(2 ** N), N)
SCORES, GRAPHS = alexandra.objective_scores(READS, STATES)
FULL = SCORES['A baseline']
MEC = SCORES['D MEC']
FULL_OPT, MEC_OPT = FULL.min(), MEC.min()
A, D, J_FULL, OFFSET = GRAPHS['A baseline']
assert np.allclose(tudor['costs'](STATES, J_FULL, OFFSET), FULL)
assert np.allclose(carla['build_evidence'](READS), (A, D))
assert np.array_equal(tudor['decode_key']('10110', 5), [0, 1, 1, 0, 1])
if np.array_equal(READS, tudor['TOY_READS']):
    assert FULL_OPT == 1
    assert {'01101', '10010'} == {''.join(map(str, x)) for x in STATES[np.isclose(FULL, FULL_OPT)]}
display(pd.DataFrame(READS).rename(columns=lambda i: f'SNP {i+1}'))
print('Full pairwise optimum:', FULL_OPT, '| full-read MEC optimum:', MEC_OPT)


,SNP 1,SNP 2,SNP 3,SNP 4,SNP 5
0,0,1,1,-1,-1
1,1,0,0,-1,-1
2,-1,1,1,0,-1
3,-1,0,0,1,-1
4,-1,-1,1,0,1
5,-1,-1,0,1,0
6,0,-1,-1,0,1
7,1,-1,-1,1,0
8,0,0,-1,-1,-1


Full pairwise optimum: 1.0 | full-read MEC optimum: 1.0


## 2. Objectives and evidence/pruning choices

All exact minimizing labels are scored with a **uniform tie policy**, never using truth to pick a favorable answer. Native objectives have different units; their native values are not compared as accuracy. The table instead rescores every objective's winners on both full common references.

Select a pairwise `OBJECTIVE` (A, B1, B2 or C) and any Carla pruning rule. MEC stays a reference-only objective. `none`, `tie`, `net`, `support`, `topk`, `mstforest`, `bridge`, `random`, `strong` remain available. Default forest pruning is illustrative, not claimed to preserve the complete objective or improve accuracy.

In [3]:
objective_rows = []
for name, native in SCORES.items():
    winners = alexandra.optimal_mask(native)
    objective_rows.append(dict(objective=name, native_min=native.min(), tied_labels=int(winners.sum()),
        full_mean=float(FULL[winners].mean()), full_optimum_rate=float(np.isclose(FULL[winners], FULL_OPT).mean()),
        mec_mean=float(MEC[winners].mean()), mec_optimum_rate=float(np.isclose(MEC[winners], MEC_OPT).mean())))
display(pd.DataFrame(objective_rows))

OBJECTIVE = 'A baseline'
PRUNE_RULE, THRESHOLD = 'mstforest', None
if OBJECTIVE not in GRAPHS:
    raise ValueError('Choose a pairwise graph; full-read MEC is reference-only.')
a, d, _, _ = GRAPHS[OBJECTIVE]
kept, dropped = carla['prune'](a, d, PRUNE_RULE, THRESHOLD, np.random.default_rng(7))
ap, dp = carla['matrices_from_edges'](N, kept)
J, offset = tudor['ising'](ap, dp)
NATIVE = tudor['costs'](STATES, J, offset)
native_opt = NATIVE.min()
reps, _ = carla['minima_reps'](ap, dp)
print('Selected:', OBJECTIVE, PRUNE_RULE, '| active edges:', np.count_nonzero(J),
      '| dropped:', len(dropped), '| active blocks:', carla['n_blocks'](N, kept))
print('Every pruned minimizer remains a full minimizer:',
      carla['pruned_min_is_full_min'](A, D, reps))
display(pd.DataFrame([dict(edge=f"{e['i']+1}-{e['j']+1}", net=e['net'], support=e['total'])
                      for e in kept]))


,objective,native_min,tied_labels,full_mean,full_optimum_rate,mec_mean,mec_optimum_rate
0,A baseline,1.0,2,1.0,1.0,1.0,1.0
1,B1 per-read,1.0,2,1.0,1.0,1.0,1.0
2,B2 per-site,1.0,2,1.0,1.0,1.0,1.0
3,C deduplicated,1.0,2,1.0,1.0,1.0,1.0
4,D MEC,1.0,2,1.0,1.0,1.0,1.0


Selected: A baseline mstforest | active edges: 4 | dropped: 5 | active blocks: 1
Every pruned minimizer remains a full minimizer: True


,edge,net,support
0,2-3,4.0,4.0
1,3-4,-4.0,4.0
2,4-5,-4.0,4.0
3,1-3,-2.0,2.0


## 3. Classical comparison and locally frozen training

Tudor's greedy start + one-bit descent uses the selected graph, 16 restarts and 128 work units per restart. Its returned best candidates are rescored on the full references. The first returned candidate seeds the warm start (clipped at 0.25); no exact winner or hidden truth seeds it.

Standard, warm-start and multi-angle use p=1 and one deterministic COBYLA restart, 60 evaluations each. This is a smoke budget, not a matched hardware advantage experiment. Optimizer convergence is reported explicitly. Parameter vectors and frozen circuits are retained in this run; replay does not retrain or submit jobs.

In [4]:
P, SHOTS, MAXITER = 1, 512, 60
local_best, LOCAL_X, local_used, local_hit = tudor['greedy_local_search'](
    J, offset, 128, 16, np.random.default_rng(17), target=native_opt)
c_seed = LOCAL_X[0].astype(float)
standard, standard_params, _ = tudor['make_qaoa'](J, P)
warm, warm_params = carla['warm_start_qaoa'](N, J, c_seed, P, eps=0.25)
multi, multi_params = carla['make_ma_qaoa'](J, P)
TEMPLATES = {'standard': (standard, list(standard_params)),
             'warm-start': (warm, warm_params), 'multi-angle': (multi, multi_params)}
FROZEN, ANGLES, training_rows = {}, {}, []
for name, (circuit, params) in TEMPLATES.items():
    frozen, result, evaluations = local_frozen_fit(circuit, params, NATIVE, MAXITER, seed=71)
    assert len(result.x) == len(params) and circuit.num_parameters <= len(params)
    assert frozen.num_parameters == 0
    FROZEN[name], ANGLES[name] = frozen, np.asarray(result.x)
    training_rows.append(dict(method=name, parameters=len(params), nfev=evaluations,
        local_expected_native=float(result.fun), converged=bool(result.success),
        optimizer_message=str(result.message), frozen_angles=np.round(result.x, 6).tolist()))
display(pd.DataFrame(training_rows))


,method,parameters,nfev,local_expected_native,converged,optimizer_message,frozen_angles
0,standard,2,34,5.280811,True,Return from COBYLA because the trust region ra...,"[2.45261, 3.534321]"
1,warm-start,2,33,1.970712,True,Return from COBYLA because the trust region ra...,"[0.382942, 0.444026]"
2,multi-angle,9,60,2.139889,False,Return from COBYLA because the objective funct...,"[1.961781, 0.74207, 2.531579, 1.757296, 0.6607..."


## 4. Frozen replay: ideal and noisy simulator

Alex's setup now accepts the common reads, selected weighted/pruned evidence and frozen standard angles without changing its original pilot defaults. Permuted logical wires are mapped back to SNP classical bits. Warm-start and multi-angle are compiled directly, preserving their different gates.

Noise uses **FakeManilaV2 snapshot + Aer**, seed 72 and 512 shots per method; these are not current calibration data or QPU results. No readout mitigation is applied to these raw samples. Alex's standalone readout/TREX/M3/ZNE/twirling/DD/layout/routing/scheduling notebooks remain available for separate investigations.

In [5]:
alex = load_hardware(READS, ANGLES['standard'][:P], ANGLES['standard'][P:], (ap, dp))
measured_standard = alex['make_circuit']()
assert np.allclose(alex['logical_probabilities'](measured_standard),
                   Statevector.from_instruction(FROZEN['standard']).probabilities())
assert np.allclose(alex['logical_probabilities'](alex['make_circuit'](tuple(reversed(range(N))))),
                   alex['logical_probabilities'](measured_standard))
noisy_sim = AerSimulator(noise_model=NoiseModel.from_backend(alex['backend']),
                         max_parallel_threads=1)
ideal_sim = AerSimulator(max_parallel_threads=1)

def samples_from_counts(counts, n):
    return np.asarray([tudor['decode_key'](key, n)
                       for key, count in counts.items() for _ in range(count)], dtype=np.int8)

def summarize_samples(method, mode, samples, **extra):
    full = alexandra.pairwise_scores(samples, A, D)
    mec = alexandra.mec_scores(samples, READS)
    native = tudor['costs'](samples, J, offset)
    return dict(method=method, mode=mode, samples=len(samples), native_mean=float(native.mean()),
        full_mean=float(full.mean()), full_optimum_rate=float(np.isclose(full, FULL_OPT).mean()),
        mec_mean=float(mec.mean()), mec_optimum_rate=float(np.isclose(mec, MEC_OPT).mean()), **extra)

rows = [summarize_samples('greedy local search', 'local classical', LOCAL_X,
                         work_per_restart=float(local_used.mean()))]
RAW_COUNTS = {}
for name, frozen in FROZEN.items():
    measured = frozen.copy()
    measured.measure_all()
    for mode, simulator in [('ideal simulator', ideal_sim), ('noisy simulator (not QPU)', noisy_sim)]:
        compiled = (transpile(measured, alex['backend'], optimization_level=1, seed_transpiler=7)
                    if mode.startswith('noisy') else transpile(measured, simulator))
        counts = simulator.run(compiled, shots=SHOTS, seed_simulator=72).result().get_counts()
        RAW_COUNTS[(name, mode)] = counts
        samples = samples_from_counts(counts, N)
        assert samples.shape == (SHOTS, N)
        direct_mean = sum(alexandra.pairwise_scores(
            tudor['decode_key'](key, N)[None, :], A, D)[0] * count
            for key, count in counts.items()) / SHOTS
        assert np.isclose(direct_mean, alexandra.pairwise_scores(samples, A, D).mean())
        rows.append(summarize_samples(name, mode, samples, **alex['resources'](compiled)))
display(pd.DataFrame(rows).round(4))
print('Raw noisy count example:', next(counts for (name, mode), counts in RAW_COUNTS.items()
                                       if mode.startswith('noisy')))
print('Real QPU execution: NOT EXECUTED; no credentials or jobs used.')


5 SNPs, 4 active interactions, backend: fake_manila
Exact minimum: 0.0
The reversed SNP order keeps the same ideal output distribution.


,method,mode,samples,native_mean,full_mean,full_optimum_rate,mec_mean,mec_optimum_rate,work_per_restart,2q gates,2q depth,depth
0,greedy local search,local classical,16,0.0000,1.0000,1.0000,1.0000,1.0000,127.25,NaN,NaN,NaN
1,standard,ideal simulator,512,5.4883,11.3867,0.0566,5.8555,0.0566,NaN,4.0,4.0,7.0
2,standard,noisy simulator (not QPU),512,5.6406,11.4785,0.0645,5.9277,0.0645,NaN,11.0,11.0,29.0
3,warm-start,ideal simulator,512,1.8242,5.0488,0.5762,2.8887,0.5762,NaN,4.0,4.0,9.0
4,warm-start,noisy simulator (not QPU),512,3.0703,6.7793,0.4414,3.7402,0.4414,NaN,11.0,11.0,30.0
5,multi-angle,ideal simulator,512,2.1758,5.5938,0.4668,3.2734,0.4668,NaN,4.0,4.0,7.0
6,multi-angle,noisy simulator (not QPU),512,3.4297,7.2129,0.3613,4.0371,0.3613,NaN,11.0,11.0,29.0


Raw noisy count example: {'10011': 28, '10010': 21, '01100': 33, '01010': 23, '10000': 43, '01110': 24, '01101': 12, '10111': 43, '01011': 15, '10101': 15, '11111': 6, '01000': 41, '00011': 6, '00101': 7, '11110': 10, '01001': 21, '11000': 18, '01111': 38, '10001': 14, '00111': 19, '00110': 7, '00000': 16, '10110': 12, '10100': 7, '11001': 13, '00100': 3, '00001': 6, '11010': 2, '11100': 1, '11011': 2, '00010': 4, '11101': 2}
Real QPU execution: NOT EXECUTED; no credentials or jobs used.


## 5. Focused integration invariants

A mixed-length counterexample checks that native objective disagreement survives full common-reference rescoring. Fractional weights must survive Carla's pruning adapter. Circuit binding is checked for standard and arbitrary-count multi-angle. The engine's optional sampled plugin trainer gets a short local Aer smoke run (not a QPU loop); failed rows are asserted rather than concealed.

In [6]:
zero_j = np.zeros((3, 3))
zero_standard, zero_params, _ = tudor['make_qaoa'](zero_j, 2)
zero_warm, warm_params = carla['warm_start_qaoa'](3, zero_j, np.zeros(3), 2, eps=0.25)
for circuit, params in [(zero_standard, list(zero_params)), (zero_warm, warm_params)]:
    frozen, fitted, _ = local_frozen_fit(circuit, params, np.zeros(8), maxiter=10)
    assert frozen.num_parameters == 0 and len(fitted.x) == len(params)
    assert np.allclose(fitted.x[:2], 0) and np.isclose(fitted.fun, 0)
print('PASS: fully pruned graphs retain full angle vectors with zero unused cost angles.')

counterexample = np.array([[0, 0, 0, 0], [0, 0, 0, 0],
                            [0, 1, -1, -1], [0, 1, -1, -1], [0, 1, -1, -1]])
counter_states = tudor['bits'](np.arange(16), 4)
cs, cg = alexandra.objective_scores(counterexample, counter_states)
wa, wb = alexandra.optimal_mask(cs['A baseline']), alexandra.optimal_mask(cs['B1 per-read'])
assert not np.array_equal(wa, wb)
assert cs['A baseline'][wb].mean() > cs['A baseline'][wa].mean()
fa, fd, _, _ = alexandra.build_graph(np.array([[0, 1, 1]]), alexandra.per_read_weight)
fk, _ = carla['prune'](fa, fd, 'none')
fpa, fpd = carla['matrices_from_edges'](3, fk)
assert np.allclose(fpa, fa) and np.allclose(fpd, fd)
_, fc = carla['exact_reference'](fpa, fpd)
assert np.allclose(fc, alexandra.pairwise_scores(alexandra.all_states(3), fa, fd))
for n, edges in [(3, 0), (3, 1), (5, 4)]:
    test_j = np.zeros((n, n))
    for i, j in list(itertools.combinations(range(n), 2))[:edges]:
        test_j[i, j] = -0.5
    qc, params = carla['make_ma_qaoa'](test_j, 2)
    assert len(params) == 2 * (edges + n)
    bound = qc.assign_parameters(dict(zip(params, np.zeros(len(params)))))
    assert bound.num_parameters == 0
    assert np.allclose(Statevector.from_instruction(bound).probabilities(), 1 / 2 ** n)

def multi_plugin(j, p):
    qc, params = carla['make_ma_qaoa'](j, p)
    return qc, params, tudor['qaoa_scale'](j)

tudor['REGISTRY']['qaoa']['multi-angle'] = multi_plugin
plugin_rows = tudor['run_instance'](tudor['toy_instance'](), tudor['Config'](
    methods=('qaoa',), qaoa='multi-angle', backend='aer_sv', trainer='cobyla',
    shots=16, train_shots=16, train_iters=5, ps=(1,), seed=4), log=lambda *_: None)
assert plugin_rows and all('fail' not in row for row in plugin_rows), plugin_rows
assert len(plugin_rows[0]['angles'].split()) == len(multi_plugin(J_FULL, 1)[1])
deterministic = QuantumCircuit(N)
pattern = np.arange(N) % 2
for q in np.flatnonzero(pattern):
    deterministic.x(int(q))
deterministic.measure_all()
dc = ideal_sim.run(transpile(deterministic, ideal_sim), shots=16).result().get_counts()
assert np.array_equal(samples_from_counts(dc, N), np.tile(pattern, (16, 1)))
print('PASS: baseline/bit order, objective disagreement/common scoring, fractional evidence,')
print('arbitrary parameter binding, sampled plugin trainer, raw-count decoding.')


PASS: fully pruned graphs retain full angle vectors with zero unused cost angles.


C:\Users\tpricop\.copilot\session-state\33083fc8-7f0c-4721-a824-fe2f9bb1a902\files\venv\Lib\site-packages\scipy\_external\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 16
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


PASS: baseline/bit order, objective disagreement/common scoring, fractional evidence,
arbitrary parameter binding, sampled plugin trainer, raw-count decoding.


## Interpretation and boundaries

Compare the **full_mean/full_optimum_rate** and **mec_mean/mec_optimum_rate** columns, not differently scaled native objective values. Forest pruning can introduce additional minima; its full-reference preservation is printed explicitly. These small budgets and one noise seed do not establish statistical superiority or quantum speedup. Saved training status includes iteration-limit failures; sampled execution/validation failures raise errors.

Standalone entry points: [Alexandra objectives](../docs/Alexandra_Objective_Variants.ipynb), [Carla lab/pruning](../docs/Quantum_Genome_Detective_Lab.ipynb), [Tudor engine](qgd_bench.ipynb), [Tudor experiment](tudor_classical_vs_quantum.ipynb), [plugin template](plugin_template.ipynb), [Alex hardware studies](../docs/hardware_experiments/README.md). Full objective archives were deliberately not imported; Alexandra's notebooks/scripts can regenerate them with their original sweeps. Optional mitigation methods were preserved, not rerun here. Real-device execution and recursive QAOA remain unimplemented/unexecuted.